In [49]:
!pip install lightning

In [50]:
from google.colab import drive

# Mount google drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [51]:
import os

# Change working directory to coding environment
os.chdir("/content/drive/Othercomputers/Laptop HP/TA/Coding")

In [52]:
from pathlib import Path

cwd = Path.cwd()
print(cwd)

/content/drive/Othercomputers/Laptop HP/TA/Coding


# Analisis karakteristik data

## Load data dari penyimpanan

In [53]:
import pandas as pd

data_dir = cwd / "data"

mortalitas_df = pd.read_csv(data_dir / "mortalitas.csv", parse_dates=["year"], date_format="%Y")
populasi_df = pd.read_csv(data_dir / "populasi.csv")
bi_rate_df = pd.read_csv(data_dir / "bi_rate.csv", parse_dates=["date"], date_format="%d-%m-%y")

In [54]:
YEAR_COL = "year"
AGE_COL = "age"
SEX_COL = "sex"
MORTALITY_COL = "mortality_rate"

In [55]:
AGE_MIN = min(mortalitas_df[AGE_COL])
AGE_MAX = max(mortalitas_df[AGE_COL])

YEAR_MIN = min(mortalitas_df[YEAR_COL])
YEAR_MAX = max(mortalitas_df[YEAR_COL])

## Analisa statistika deskriptif

In [56]:
mortalitas_df.head()

,year,sex,age,mortality_rate
0,1950-01-01,Male,0,0.228515
1,1950-01-01,Female,0,0.185158
2,1950-01-01,Male,1,0.069349
3,1950-01-01,Female,1,0.063058
4,1950-01-01,Male,2,0.044724


In [57]:
mortalitas_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15150 entries, 0 to 15149
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   year            15150 non-null  datetime64[ns]
 1   sex             15150 non-null  object        
 2   age             15150 non-null  int64         
 3   mortality_rate  15150 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 473.6+ KB


In [58]:
populasi_df.head()

,sex,age,value
0,Male,45,2013106
1,Male,46,1965078
2,Male,47,1918556
3,Male,48,1880902
4,Male,49,1836713


In [59]:
populasi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   sex     32 non-null     object
 1   age     32 non-null     int64 
 2   value   32 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 900.0+ bytes


In [60]:
bi_rate_df.head()

,date,bi_rate
0,2024-12-18,0.0600
1,2024-11-20,0.0600
2,2024-10-16,0.0600
3,2024-09-18,0.0600
4,2024-08-21,0.0625


In [61]:
bi_rate_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     60 non-null     datetime64[ns]
 1   bi_rate  60 non-null     float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 1.1 KB


## Plot mortalitas satu usia untuk semua tahun

In [62]:
import seaborn as sns

sns.set_theme(style="whitegrid")

In [63]:
plot_dir = cwd / "plot"

In [64]:
from ta_module.utils import plot_usia_vs_tahun


def _wrapper_plot_usia_vs_tahun(age_start: int, age_end: int):
    plot_usia_vs_tahun(
        mortalitas_df=mortalitas_df,
        age_col=AGE_COL,
        year_col=YEAR_COL,
        sex_col=SEX_COL,
        mortality_col=MORTALITY_COL,
        age_start=age_start,
        age_end=age_end,
        plot_dir=plot_dir
    )

In [65]:
_wrapper_plot_usia_vs_tahun(AGE_MIN, 20)
_wrapper_plot_usia_vs_tahun(21, 40)
_wrapper_plot_usia_vs_tahun(41, 60)
_wrapper_plot_usia_vs_tahun(61, 80)
_wrapper_plot_usia_vs_tahun(81, AGE_MAX)

## Plot mortalitas semua usia untuk satu tahun

In [66]:
from ta_module.utils import plot_tahun_vs_usia


def _wrapper_plot_tahun_vs_usia(year_start: str, year_end: str):
    plot_tahun_vs_usia(
        mortalitas_df=mortalitas_df,
        age_col=AGE_COL,
        year_col=YEAR_COL,
        sex_col=SEX_COL,
        mortality_col=MORTALITY_COL,
        year_start=year_start,
        year_end=year_end,
        plot_dir=plot_dir
    )

In [67]:
_wrapper_plot_tahun_vs_usia("1950", "1964")
_wrapper_plot_tahun_vs_usia("1965", "1979")
_wrapper_plot_tahun_vs_usia("1980", "1994")
_wrapper_plot_tahun_vs_usia("1995", "2009")
_wrapper_plot_tahun_vs_usia("2010", "2024")

# Pemodelan LocalGLMnet

In [108]:
from lightning.pytorch import seed_everything

# Atur seed agar reproducible
# seed yang diatur termasuk numpy, pytorch, dan python random number generator
seed_everything(42, workers=True)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


42

## Persiapan data untuk pelatihan

### Matriks mortalitas (M)

In [109]:
# Male mortalitas
mortalitas_df_male = mortalitas_df[mortalitas_df[SEX_COL] == "Male"]
M_male = mortalitas_df_male.pivot(
    index=YEAR_COL,
    columns=AGE_COL,
    values=MORTALITY_COL
)

# Female mortalitas
mortalitas_df_female = mortalitas_df[mortalitas_df[SEX_COL] == "Female"]
M_female = mortalitas_df_female.pivot(
    index=YEAR_COL,
    columns=AGE_COL,
    values=MORTALITY_COL
)

### Train-Val-Test split

In [ ]:
from ta_module.data import get_train_val_test_split

train_split, val_split, test_split = (0.8, 0.1, 0.1)
M_male_train, M_male_val, M_male_test = get_train_val_test_split(
    df=M_male,
    train_split=train_split,
    val_split=val_split,
    test_split=test_split
)

M_female_train, M_female_val, M_female_test = get_train_val_test_split(
    df=M_female,
    train_split=train_split,
    val_split=val_split,
    test_split=test_split
)

In [ ]:
import torch
import numpy as np

from ta_module.data import NormalizedMortalityDataset

train_male_mean = torch.from_numpy(M_male_train.mean(axis=1).to_numpy(copy=True, dtype=np.float32))
train_male_std = torch.from_numpy(M_male_train.std(axis=1).to_numpy(copy=True, dtype=np.float32))

lookback = 10
horizon = 1

create_male_dataset_split = NormalizedMortalityDataset.factory(
    lookback=lookback,
    horizon=horizon,
    mean=train_male_mean,
    std=train_male_std
)

train_male_dataset = create_male_dataset_split(M_male_train)
val_male_dataset = create_male_dataset_split(M_male_val)
test_male_dataset = create_male_dataset_split(M_male_test)

In [ ]:
train_female_mean = torch.from_numpy(M_female_train.mean(axis=1).to_numpy(copy=True, dtype=np.float32))
train_female_std = torch.from_numpy(M_female_train.std(axis=1).to_numpy(copy=True, dtype=np.float32))

create_female_dataset_split = NormalizedMortalityDataset.factory(
    lookback=lookback,
    horizon=horizon,
    mean=train_female_mean,
    std=train_female_std
)

train_female_dataset = create_female_dataset_split(M_female_train)
val_female_dataset = create_female_dataset_split(M_female_val)
test_female_dataset = create_female_dataset_split(M_female_test)

In [ ]:
from torch.utils.data import ConcatDataset

train_dataset = ConcatDataset([train_male_dataset, train_female_dataset])
val_dataset = ConcatDataset([val_male_dataset, val_female_dataset])
test_dataset = ConcatDataset([test_male_dataset, test_female_dataset])

In [ ]:
from torch.utils.data import DataLoader

batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

## Definisi arsitektur model

### LCN

In [111]:
from torch import nn
from ta_module.models import LocalGLMnet, LocallyConnected2D

ukuran_matriks_input = (lookback, AGE_MAX - AGE_MIN + 1)
lcn_activation_function = nn.Sigmoid()
lcn_kernel_size = 5

create_lcn_layer = LocallyConnected2D.factory(
    input_size=ukuran_matriks_input,
    kernel_size=lcn_kernel_size,
    activation_fn=lcn_activation_function,
    zero_padding=True,
    bias=True
)

### LocalGLMnet

In [112]:
from ta_module.utils import IdentityTransform

create_localglmnet_model = LocalGLMnet.factory(
    input_size=ukuran_matriks_input,
    link_fn=IdentityTransform(),
    bias=True
)

In [113]:
num_ensembles = 10
localglmnet_models = [
    create_localglmnet_model(regression_attention_model=create_lcn_layer())
    for _ in range(num_ensembles)
]

### Loss function

In [114]:
import torch.nn.functional as F

from ta_module.utils import RegularizationLoss

train_loss_fn = F.mse_loss
eval_loss_fn = F.mse_loss

regularization_parameter = 1e-5
create_regularization_loss = RegularizationLoss.factory(
    eta=regularization_parameter,
    # alfa = 1 -> LASSO regularization
    alfa=1
)

### Optimizer

In [115]:
from torch.optim import NAdam
from typing import Iterator


def _nadam_optimizer_factory(*args, **kwargs):
    def create_nadam_optimizer(params: Iterator[nn.Parameter]):
        return NAdam(
            params,
            *args,
            **kwargs
        )

    return create_nadam_optimizer

optimizer_factory = _nadam_optimizer_factory(lr=1e-3)

### Ensemble dan trainer

In [117]:
from ta_module.models import MyModel

loss_log_scale = 4
localglmnet_ensembles = [
    MyModel(
        model=m,
        train_loss=train_loss_fn,
        eval_loss=eval_loss_fn,
        regularization_loss=create_regularization_loss(
            model_weights_getter=(lambda: m.regression_attention_model.parameters())
        ),
        optimizer_factory=optimizer_factory,
        loss_log_scale=loss_log_scale
    )

    for m in localglmnet_models
]

In [118]:
import lightning as L
import os

from datetime import datetime
from zoneinfo import ZoneInfo
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger

timezone = ZoneInfo("Asia/Jakarta")
run_datetime = datetime.now(tz=timezone).strftime("%Y-%m-%d_%H-%M-%S")

checkpoint_dirs = [f"best_model_checkpoints/LocalGLMnet_{i + 1}" for i in range(num_ensembles)]
checkpoint_filename = f"{run_datetime}"

log_dir_tensorboard = "logs_tensorboard"
log_dir_csv = "logs_csv"

os.makedirs(log_dir_tensorboard, exist_ok=True)
os.makedirs(log_dir_csv, exist_ok=True)
for d in checkpoint_dirs:
    os.makedirs(d, exist_ok=True)

num_epochs = 500
trainers = [
    L.Trainer(
        max_epochs=num_epochs,
        callbacks=[
            ModelCheckpoint(
                dirpath=checkpoint_dirs[i],
                filename=checkpoint_filename,
                monitor="val_loss",
                mode="min",
                save_top_k=1
            ),
            EarlyStopping(
                min_delta=1e-5,
                monitor="val_loss",
                mode="min",
                patience=5,
            )
        ],
        logger=[
            TensorBoardLogger(
                save_dir=log_dir_tensorboard,
                name=checkpoint_filename[i],
                version=run_datetime,
            ),
            CSVLogger(
                save_dir=log_dir_csv,
                name=checkpoint_dirs[i],
                version=run_datetime
            )
        ],
        log_every_n_steps=1,
        deterministic=True
    )
    for i in range(num_ensembles)
]

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: F

In [119]:
for i, (trainer, model) in enumerate(zip(trainers, localglmnet_ensembles)):
    print("\n=============================")
    print(i)
    print(trainer)
    print(model)


0
MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
  (regularization_loss): RegularizationLoss()
)

1
MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
  (regularization_loss): RegularizationLoss()
)

2
MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
  (regularization_loss): RegularizationLoss()
)

3
MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
  (regularization_loss): RegularizationLoss()
)

4
MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
  (regularization_loss): RegularizationLoss()
)

5
MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
     

## Proses pelatihan

In [1]:
import json

# Train ensembles pada data mortalitas laki-laki dan perempuan
print(f"Melatih {num_ensembles} model LocalGLMnet secara independen")
print(f"Run datetime: {run_datetime}")
for i, (trainer, model) in enumerate(zip(trainers, localglmnet_ensembles)):
    print("\n================================================================")
    print(f"Training model {i + 1} pada mortalitas laki-laki dan perempuan:")
    print("================================================================\n")
    trainer.fit(
        model=model,
        train_dataloader=train_dataloader,
        val_dataloaders=val_dataloader,
        test_dataloaders=test_dataloader,
    )

print("Pelatihan selesai!")

# metadata digunakan untuk load checkpoint model terakhir secara otomatis
# jika tidak run proses pelatihan
metadata = {
    "last_run_datetime"    : run_datetime,
    "num_of_models"        : num_ensembles,
    "checkpoint_file_paths": [
        f"{checkpoint_dirs[i]}/{checkpoint_filename}.ckpt"
        for i in range(num_ensembles)
    ]
}

print("Update metadata last_run.json:")
with open("last_run.json", "w") as f:
    json.dump(metadata, f, indent=4)
print("Update berhasil!")

NameError: name 'run_datetime' is not defined

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir logs_tensorboard

# Interpretasi pemodelan

## Load best model dari checkpoint

In [ ]:
import json
import os


def load_last_checkpoint():
    if not os.path.exists("last_run.json"):
        return None

    with open("last_run.json") as f:
        metadata = json.load(f)

    last_run_datetime = metadata["last_run_datetime"]
    return last_run_datetime

In [ ]:
def get_last_run_time():
    metadata_file = "last_run.json"
    if not os.path.exists(metadata_file):
        return None

    with open(metadata_file) as f:
        metadata = json.load(f)

    return metadata["last_run_datetime"]

In [122]:
last_run_datetime = get_last_run_time()
best_model_checkpoints_dirs = [f"{checkpoint_dirs[i]}/{last_run_datetime}.ckpt" for i in range(num_ensembles)]
localglmnet_ensembles_best = [
    MyModel.load_from_checkpoint(
        c,
        weights_only=False,
        model=m,
        train_loss=train_loss_fn,
        eval_loss=eval_loss_fn,
        regularizaton_loss=create_regularization_loss(
            model_weights_getter=(lambda: m.regression_attention_model.parameters())
        ),
        optimizer_factory=optimizer_factory,
        lr_scheduler_factory=lr_scheduler_factory
    )

    for c, m in zip(
        best_model_checkpoints_dirs,
        [create_localglmnet_model(regression_attention_model=create_lcn_layer()) for _ in
         range(num_ensembles)]
    )
]

In [123]:
print(localglmnet_ensembles_best)

[MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): LocallyConnected2D(
      (activation_fn): Sigmoid()
    )
  )
), MyModel(
  (model): LocalGLMnet(
    (regression_attention_model): L

In [125]:
import torch

checkpoints = [
    torch.load(f"{checkpoint_dir}/{f}.ckpt")
    for f in checkpoint_filenames
]

print("Cek apakah parameter di localglmnet_ensembles_best sama dengan parameter di checkpoint:")
for i, (c, m_1) in enumerate(zip(checkpoints, localglmnet_ensembles_best)):
    print(f"\nModel {i + 1}")
    print("=============================================")
    checkpoint_state_dict = c["state_dict"]
    for n_1, p_1 in m_1.named_parameters():
        print(f"{n_1}:", torch.equal(checkpoint_state_dict[n_1], p_1))
    print("=============================================")

Cek apakah parameter di localglmnet_ensembles_best sama dengan parameter di checkpoint:

Model 1
model.bias: True
model.regression_attention_model.weight: True
model.regression_attention_model.bias: True

Model 2
model.bias: True
model.regression_attention_model.weight: True
model.regression_attention_model.bias: True

Model 3
model.bias: True
model.regression_attention_model.weight: True
model.regression_attention_model.bias: True

Model 4
model.bias: True
model.regression_attention_model.weight: True
model.regression_attention_model.bias: True

Model 5
model.bias: True
model.regression_attention_model.weight: True
model.regression_attention_model.bias: True

Model 6
model.bias: True
model.regression_attention_model.weight: True
model.regression_attention_model.bias: True

Model 7
model.bias: True
model.regression_attention_model.weight: True
model.regression_attention_model.bias: True

Model 8
model.bias: True
model.regression_attention_model.weight: True
model.regression_attention_m